In [1]:
import requests
import httpx  # if you want to use httpx instead of requests

import pandas as pd
from bs4 import BeautifulSoup
from time import sleep

In [2]:
url="https://www.unitedstateszipcodes.org/"
# Your code here
html_page = requests.get(url)

In [3]:
html_page.status_code


403

In [4]:
 html_page.text[:2000] # First 2000 characters of the HTML

'<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">\n<HTML><HEAD><META HTTP-EQUIV="Content-Type" CONTENT="text/html; charset=iso-8859-1">\n<TITLE>ERROR: The request could not be satisfied</TITLE>\n</HEAD><BODY>\n<H1>403 ERROR</H1>\n<H2>The request could not be satisfied.</H2>\n<HR noshade size="1px">\nRequest blocked.\nWe can\'t connect to the server for this app or website at this time. There might be too much traffic or a configuration error. Try again later, or contact the app or website owner.\n<BR clear="all">\nIf you provide content to customers through CloudFront, you can find steps to troubleshoot and help prevent this error by reviewing the CloudFront documentation.\n<BR clear="all">\n<HR noshade size="1px">\n<PRE>\nGenerated by cloudfront (CloudFront)\nRequest ID: dwv6jyG-OACcoAzZirSkRA-L4QxRiOZo8TGJ84fk4kyTTEwn4vXUpA==\n</PRE>\n<ADDRESS>\n</ADDRESS>\n</BODY></HTML>'

In [5]:
soup = BeautifulSoup(html_page.text, 'html.parser')


In [7]:
soup.find('header')

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 1. رابط الموقع المراد سحب البيانات منه
url = 'https://www.unitedstateszipcodes.org/'

# 2. إرسال طلب للجلب (مع إخفاء الهوية كمتصفح عادي لتجنب الحظر)
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)

# 3. التحقق من نجاح الاتصال (200 تعني تم)
if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    
    data = []
    # 4. البحث عن العناصر (مثال: البحث عن كل المكونات داخل div له class معينة)
    items = soup.find_all('div', class_='item-class')
    
    for item in items:
        # استخراج النص مع تنظيف المسافات
        title = item.find('h2').text.strip() if item.find('h2') else None
        price = item.find('span', class_='price').text.strip() if item.find('span', class_='price') else None
        
        data.append({'Title': title, 'Price': price})
    
    # 5. تحويل البيانات إلى الجدول الآتي للتحليل
    df = pd.DataFrame(data)
    print(df.head())
else:
    print(f"فشل الاتصال: {response.status_code}")

In [9]:
df1.columns

Index(['id', 'date', 'price', 'bedrooms', 'bathrooms', 'sqft_living',
       'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade',
       'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode',
       'lat', 'long', 'sqft_living15', 'sqft_lot15'],
      dtype='object')

In [10]:
df2.columns

Index(['zip', 'type', 'decommissioned', 'primary_city', 'acceptable_cities',
       'unacceptable_cities', 'state', 'county', 'timezone', 'area_codes',
       'world_region', 'country', 'latitude', 'longitude',
       'irs_estimated_population'],
      dtype='object')

In [12]:
df2.isnull().sum()

zip                             0
type                            0
decommissioned                  0
primary_city                    0
acceptable_cities           33433
unacceptable_cities         31062
state                           0
county                        929
timezone                      791
area_codes                    816
world_region                42402
country                        78
latitude                        0
longitude                       0
irs_estimated_population        0
dtype: int64

In [19]:
import pandas as pd

# 1. قراءة الملفين (استبدلي الأسباب بأسماء ملفاتك الحقيقية)
df1 = pd.read_csv('./Data/kc_house_data.csv')  # الملف الأول (مثلاً الملف الذي يحتوي على بياناتك الأساسية)
df2 = pd.read_csv('Data/zip_code_database.csv')  # الملف الثاني (الذي يحتوي على الرمز البريدي والمنطقة)

# 2. اختيار الأعمدة التي تريدينها فقط من كل ملف
# تفقدي أسماء الأعمدة جيداً واكتبيها بدقة
df1_selected = df1[['id', 'date', 'price', 'bedrooms', 'bathrooms', 'sqft_living',
       'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade',
       'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode',
       'lat', 'long', 'sqft_living15', 'sqft_lot15']]  # حددي الأعمدة المطلوبة من الملف الأول (مع عمود الـ zip)
df2_selected = df2[['zip', 'primary_city', 'state','country']] # حددي عمود الـ zip وعمود المنطقة/المدينة من الملف الثاني

# 3. دمج الملفين بناءً على عمود الرمز البريدي (zip)
# how='left' تضمن الاحتفاظ بكل صفوف الملف الأول وإضافة المنطقة المقابلة لها
merged_df = pd.merge(df1_selected, df2_selected, left_on='zipcode', right_on='zip', how='left')
# 4. معاينة النتيجة
print(merged_df.head())

# 5. حفظ النتيجة في ملف جديد (حسب الرغبة)
merged_df.to_csv('merged_output.csv', index=False)

           id             date     price  bedrooms  bathrooms  sqft_living  \
0  7129300520  20141013T000000  221900.0         3       1.00         1180   
1  6414100192  20141209T000000  538000.0         3       2.25         2570   
2  5631500400  20150225T000000  180000.0         2       1.00          770   
3  2487200875  20141209T000000  604000.0         4       3.00         1960   
4  1954400510  20150218T000000  510000.0         3       2.00         1680   

   sqft_lot  floors  waterfront  view  ...  yr_renovated  zipcode      lat  \
0      5650     1.0           0     0  ...             0    98178  47.5112   
1      7242     2.0           0     0  ...          1991    98125  47.7210   
2     10000     1.0           0     0  ...             0    98028  47.7379   
3      5000     1.0           0     0  ...             0    98136  47.5208   
4      8080     1.0           0     0  ...             0    98074  47.6168   

      long  sqft_living15  sqft_lot15    zip  primary_city  st

In [20]:
merged_df.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,zip,primary_city,state,country
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,0,98178,47.5112,-122.257,1340,5650,98178,Seattle,WA,US
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,1991,98125,47.7210,-122.319,1690,7639,98125,Seattle,WA,US
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,0,98028,47.7379,-122.233,2720,8062,98028,Kenmore,WA,US
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,0,98136,47.5208,-122.393,1360,5000,98136,Seattle,WA,US
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,0,98074,47.6168,-122.045,1800,7503,98074,Sammamish,WA,US


In [21]:
merged_df.isnull().sum()

id               0
date             0
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
grade            0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
zipcode          0
lat              0
long             0
sqft_living15    0
sqft_lot15       0
zip              0
primary_city     0
state            0
country          0
dtype: int64

In [22]:
print(merged_df['country'].nunique())

1


In [23]:
merged_df.drop(columns=['country'], inplace=True)

In [27]:
merged_df.drop(columns=['zip'], inplace=True)

In [30]:
merged_df.drop(columns=['state'], inplace=True)

In [31]:
merged_df.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,primary_city
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,1180,0,1955,0,98178,47.5112,-122.257,1340,5650,Seattle
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639,Seattle
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,770,0,1933,0,98028,47.7379,-122.233,2720,8062,Kenmore
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,1050,910,1965,0,98136,47.5208,-122.393,1360,5000,Seattle
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,1680,0,1987,0,98074,47.6168,-122.045,1800,7503,Sammamish


In [29]:
print(merged_df['state'].nunique())

1


In [33]:
merged_df.columns

Index(['id', 'date', 'price', 'bedrooms', 'bathrooms', 'sqft_living',
       'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade',
       'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode',
       'lat', 'long', 'sqft_living15', 'sqft_lot15', 'primary_city'],
      dtype='object')

In [34]:
merged_df['primary_city'].nunique()

24

In [35]:
print(merged_df.isnull().sum())
print(merged_df.duplicated().sum())

id               0
date             0
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
grade            0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
zipcode          0
lat              0
long             0
sqft_living15    0
sqft_lot15       0
primary_city     0
dtype: int64
0
